# 04_structured_output

Stage 4 of the `codes/mcp/` track: **structured tool output** on the same
**FastMCP + Pydantic** stack as stages 1–3.

Notebook path: `codes/mcp/04_structured_output/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`@mcp.tool` returns a **Pydantic model** → `tools/list` shows `output_schema`
→ `tools/call` fills `structured_content` (and a JSON text fallback).

Server-initiated sampling / imperative elicitation need a back-channel the
`2026-07-28` era dropped (SEP-2577 / SEP-2322). Structured output stays
current — return typed models, let FastMCP own the wire shape.

**Boundary:** this pattern fits **Python-authored** FastMCP tools. Tools
exported from C++ (e.g. CppReflectKit) only benefit from `output_schema` if
the Python side also hand-writes a matching result model — often not worth
it; reflect-kit should not force structured_output. This stage is teaching,
not a CppReflectKit prescription.

FastMCP's Client is async — cells below use top-level `await` (Jupyter /
IPython). From a plain script, wrap the same coroutine in `asyncio.run(...)`.


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "04_structured_output" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "04_structured_output"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/04_structured_output")

ROOT = lab_root()
print(ROOT)


/workspace/AEFS-Notes/codes/mcp/04_structured_output


## 1. Read a host-facing tools/list shape (outputSchema)

`payload` is what a host sees after listing tools. Structured-output tools
advertise both `inputSchema` and `outputSchema`.

Print every tool `name`, then print the `outputSchema` property keys of
`sum_pair`. Confirm `total` and `expression` are present.


In [2]:
payload = {
    "tools": [
        {
            "name": "sum_pair",
            "description": "Add two numbers and return a structured SumResult.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "a": {"description": "Left operand", "type": "number"},
                    "b": {"description": "Right operand", "type": "number"},
                },
                "required": ["a", "b"],
            },
            "outputSchema": {
                "type": "object",
                "properties": {
                    "total": {"description": "a + b", "type": "number"},
                    "expression": {
                        "description": "Human-readable a + b = total",
                        "type": "string",
                    },
                },
                "required": ["total", "expression"],
            },
        }
    ]
}

# answer

names = [t["name"] for t in payload["tools"]]
print(names)
sum_tool = next(t for t in payload["tools"] if t["name"] == "sum_pair")
out_props = sorted(sum_tool["outputSchema"]["properties"])
print(out_props)
assert out_props == ["expression", "total"]


['sum_pair']
['expression', 'total']


## 2. Tiny FastMCP server that returns a Pydantic model

Create a local `FastMCP("qa-struct")` with a Pydantic model `Pair` (`left`,
`right`: floats) and one tool `make_pair(left, right) -> Pair` that returns
`Pair(left=left, right=right)`.

Drive it with `Client(server)`. Print:

1. negotiated `protocol_version` (expect `2026-07-28`)
2. `make_pair.output_schema` property keys
3. `structured_content` from `call_tool("make_pair", {"left": 1.5, "right": 2.5})`


In [3]:
from typing import Annotated

from fastmcp import Client, FastMCP
from pydantic import BaseModel, Field

# answer

class Pair(BaseModel):
    left: Annotated[float, Field(description="Left value")]
    right: Annotated[float, Field(description="Right value")]

server = FastMCP("qa-struct")

@server.tool
def make_pair(
    left: Annotated[float, Field(description="Left value")],
    right: Annotated[float, Field(description="Right value")],
) -> Pair:
    """Build a structured Pair."""
    return Pair(left=left, right=right)

async with Client(server) as client:
    print("protocol:", client.protocol_version)
    tools = await client.list_tools()
    schema = tools[0].output_schema or {}
    print("output_keys:", sorted(schema.get("properties", {})))
    result = await client.call_tool("make_pair", {"left": 1.5, "right": 2.5})
    print("structured:", result.structured_content)


protocol: 2026-07-28
output_keys: ['left', 'right']


structured: {'left': 1.5, 'right': 2.5}


## 3. Drive the lab server: list → call with structured_content

Import `mcp` from `server.py` in this directory. With `Client(mcp)`:

1. print `protocol_version` and `server_info.name`
2. `list_tools` — print names and each `output_schema` property-key list
3. `call_tool("sum_pair", {"a": 2, "b": 3})` — print `structured_content`
4. `call_tool("label_topic", {"topic": "MCP"})` — print `structured_content`

Same Client entry as stages 1–3 — only the result surface grows from text to
typed JSON the host can validate against `output_schema`.


In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from server import mcp

# answer

async with Client(mcp) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    tools = await client.list_tools()
    print("tools:", [t.name for t in tools])
    for t in tools:
        keys = sorted((t.output_schema or {}).get("properties", {}))
        print(f"  {t.name}:", keys)
    sum_result = await client.call_tool("sum_pair", {"a": 2, "b": 3})
    print("sum:", sum_result.structured_content)
    label_result = await client.call_tool("label_topic", {"topic": "MCP"})
    print("label:", label_result.structured_content)


protocol: 2026-07-28
server: lab-structured-output
tools: ['sum_pair', 'label_topic']
  sum_pair: ['expression', 'total']
  label_topic: ['confidence', 'label', 'topic']
sum: {'total': 5.0, 'expression': '2.0 + 3.0 = 5.0'}
label: {'label': 'protocol', 'confidence': 0.9, 'topic': 'MCP'}


## 4. Plain dict returns also become structured_content

Build a local `FastMCP("qa-dict")` tool `stats(values: list[int]) -> dict`
that returns `{"count": len(values), "sum": sum(values)}`. Call it with
`[1, 2, 3]` and print `structured_content`.

Extension point for later stages: prefer returning Pydantic models when you
want a precise `output_schema`; a plain dict still ships as
`structured_content` with a loose object schema. Skip deprecated
server-initiated sampling in the `2026-07-28` era — call an LLM in-process
if a tool truly needs generation.


In [5]:
# answer

ext = FastMCP("qa-dict")

@ext.tool
def stats(
    values: Annotated[list[int], Field(description="Ints to summarize")],
) -> dict:
    """Return count + sum as a plain dict."""
    return {"count": len(values), "sum": sum(values)}

async with Client(ext) as client:
    result = await client.call_tool("stats", {"values": [1, 2, 3]})
    print(result.structured_content)
    assert result.structured_content == {"count": 3, "sum": 6}


{'count': 3, 'sum': 6}
